In [201]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [202]:
df_commercial_activity = pd.read_csv('data/datasets_TFM + diccionario/customer_commercial_activity.csv', sep=',')
df_sociodemographics = pd.read_csv('data/datasets_TFM + diccionario/customer_sociodemographics.csv', sep=',')
df_customer_products = pd.read_csv('data/datasets_TFM + diccionario/customer_products.csv', sep=',')
df_sales = pd.read_csv('data/datasets_TFM + diccionario/sales.csv', sep=',')

df_product_description = pd.read_csv('data/datasets_TFM + diccionario/product_description.csv', sep=',')

In [203]:
df_commercial_activity.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
df_sociodemographics.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
df_customer_products.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
df_sales.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
df_product_description.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

# 1. Preprocessing + FE

### CUSTOMER DATA

In [204]:
df = df_commercial_activity.merge(df_sociodemographics, how='left', on = ['pk_cid','pk_partition'])
df = df.merge(df_customer_products, how='left', on = ['pk_cid','pk_partition'])
df

,pk_cid,pk_partition,entry_date,entry_channel,active_customer,segment,country_id,region_code,gender,age,...,long_term_deposit,em_account_pp,credit_card,payroll,pension_plan,payroll_account,emc_account,debit_card,em_account_p,em_acount
0,1375586,2018-01,2018-01,KHL,1.0,02 - PARTICULARES,ES,29.0,H,35,...,0,0,0,0.0,0.0,0,0,0,0,1
1,1050611,2018-01,2015-08,KHE,0.0,03 - UNIVERSITARIO,ES,13.0,V,23,...,0,0,0,0.0,0.0,0,0,0,0,1
2,1050612,2018-01,2015-08,KHE,0.0,03 - UNIVERSITARIO,ES,13.0,V,23,...,0,0,0,0.0,0.0,0,0,0,0,1
3,1050613,2018-01,2015-08,KHD,0.0,03 - UNIVERSITARIO,ES,50.0,H,22,...,0,0,0,0.0,0.0,0,0,0,0,0
4,1050614,2018-01,2015-08,KHE,1.0,03 - UNIVERSITARIO,ES,50.0,V,23,...,0,0,0,0.0,0.0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5962919,1166765,2019-05,2016-08,KHE,0.0,03 - UNIVERSITARIO,ES,50.0,V,22,...,0,0,0,0.0,0.0,0,0,0,0,1
5962920,1166764,2019-05,2016-08,KHE,0.0,03 - UNIVERSITARIO,ES,26.0,V,23,...,0,0,0,0.0,0.0,0,0,0,0,1
5962921,1166763,2019-05,2016-08,KHE,1.0,02 - PARTICULARES,ES,50.0,H,47,...,0,0,0,0.0,0.0,0,0,0,0,1
5962922,1166789,2019-05,2016-08,KHE,0.0,03 - UNIVERSITARIO,ES,50.0,H,22,...,0,0,0,0.0,0.0,0,0,0,0,1


In [205]:
print("Número de registros: {}".format(df.pk_cid.count()))
print("Número de ids únicos: {}".format(df.pk_cid.nunique()))

Número de registros: 5962924
Número de ids únicos: 456373


In [206]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5962924 entries, 0 to 5962923
Data columns (total 27 columns):
 #   Column              Dtype  
---  ------              -----  
 0   pk_cid              int64  
 1   pk_partition        object 
 2   entry_date          object 
 3   entry_channel       object 
 4   active_customer     float64
 5   segment             object 
 6   country_id          object 
 7   region_code         float64
 8   gender              object 
 9   age                 int64  
 10  deceased            object 
 11  salary              float64
 12  short_term_deposit  int64  
 13  loans               int64  
 14  mortgage            int64  
 15  funds               int64  
 16  securities          int64  
 17  long_term_deposit   int64  
 18  em_account_pp       int64  
 19  credit_card         int64  
 20  payroll             float64
 21  pension_plan        float64
 22  payroll_account     int64  
 23  emc_account         int64  
 24  debit_card          int6

In [245]:
df["pk_cid"] = df["pk_cid"].astype(str)

#### Imputación de nulos

In [207]:
df.select_dtypes(include=['number']).isnull().sum().sort_values(ascending=False)

salary                1541104
region_code              2264
pension_plan               61
payroll                    61
pk_cid                      0
em_account_pp               0
em_account_p                0
debit_card                  0
emc_account                 0
payroll_account             0
credit_card                 0
long_term_deposit           0
active_customer             0
securities                  0
funds                       0
mortgage                    0
loans                       0
short_term_deposit          0
age                         0
em_acount                   0
dtype: int64

In [208]:
df.select_dtypes(include=['object']).isnull().sum().sort_values(ascending=False)

segment          133944
entry_channel    133033
gender               25
pk_partition          0
entry_date            0
country_id            0
deceased              0
dtype: int64

In [209]:
cols_with_nulls = ['salary','region_code']

for col in cols_with_nulls:
    null_pct = (df[col].isnull().sum() / len(df)) * 100
    print(f'% de nulos en {col}: {null_pct:.2f}%')

% de nulos en salary: 25.84%
% de nulos en region_code: 0.04%


In [210]:
df["salary"].describe()

count    4.421820e+06
mean     1.155833e+05
std      2.000066e+05
min      1.202730e+03
25%      6.141532e+04
50%      8.844147e+04
75%      1.313092e+05
max      2.889440e+07
Name: salary, dtype: float64

In [211]:
print("Salario medio:",df.groupby('pk_partition')["salary"].mean())
print("Salario mediana:",df.groupby('pk_partition')["salary"].median())

Salario medio: pk_partition
2018-01    112895.921277
2018-02    112898.096583
2018-03    113014.244875
2018-04    113035.403641
2018-05    113163.092118
2018-06    113254.941003
2018-07    117002.514965
2018-08    116590.408860
2018-09    116501.778626
2018-10    116488.825331
2018-11    116487.545163
2018-12    116493.811517
2019-01    116459.138496
2019-02    116453.510931
2019-03    116454.999778
2019-04    116455.702046
2019-05    116461.103930
Name: salary, dtype: float64
Salario mediana: pk_partition
2018-01    87375.390
2018-02    87429.030
2018-03    87471.150
2018-04    87535.020
2018-05    87568.950
2018-06    87598.140
2018-07    89105.100
2018-08    88864.455
2018-09    88821.300
2018-10    88813.455
2018-11    88804.065
2018-12    88803.840
2019-01    88811.595
2019-02    88811.400
2019-03    88806.090
2019-04    88803.840
2019-05    88796.610
Name: salary, dtype: float64


In [212]:
# 1. Variables numericas

# 1.1 salario
median_salary_cid = df.groupby('pk_cid')['salary'].median()
median_salary_partition = df.groupby('pk_partition')['salary'].median()

df['salary'] = df['salary'].fillna(df['pk_cid'].map(median_salary_cid))
df['salary'] = df['salary'].fillna(df['pk_partition'].map(median_salary_partition))
df['salary'] = df['salary'].fillna(df['salary'].median())  # fallback global

# 1.2 region_code (no es realmente numerica)
mode_region_cid = df.groupby('pk_cid')['region_code'].agg(lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan)
mode_region_es = df[df['country_id'] == 'ES'].groupby('pk_partition')['region_code'].agg(lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan)
mode_region_no_es = df[df['country_id'] != 'ES'].groupby('pk_partition')['region_code'].agg(lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan)

def fill_region(row):
    if pd.notna(row['region_code']):
        return row['region_code']
    if pd.notna(mode_region_cid.get(row['pk_cid'])):
        return mode_region_cid[row['pk_cid']]
    if row['country_id'] == 'ES':
        return mode_region_es.get(row['pk_partition'], np.nan)
    return mode_region_no_es.get(row['pk_partition'], np.nan)

df['region_code'] = df.apply(fill_region, axis=1)


# 1.3 para todo producto que no tenga valor, lo imputamos como 0
product_cols = ['short_term_deposit', 'loans', 'mortgage',
       'funds', 'securities', 'long_term_deposit', 'em_account_pp',
       'credit_card', 'payroll', 'pension_plan', 'payroll_account',
       'emc_account', 'debit_card', 'em_account_p', 'em_acount']

for col in product_cols:
    df[col] = df[col].fillna(0)


In [213]:
# 2. Variables categóricas

# 2.1 rellenamos los pocos nulos que hay con la moda de su partición
mode_segment = df.groupby('pk_partition')['segment'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
mode_channel = df.groupby('pk_partition')['entry_channel'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)

df['segment'] = df['segment'].fillna(df['pk_partition'].map(mode_segment))
df['entry_channel'] = df['entry_channel'].fillna(df['pk_partition'].map(mode_channel)) 

# 2.2 geneder
mode_gender_cid = df.groupby('pk_cid')['gender'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
mode_gender_partition = df.groupby('pk_partition')['gender'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)

df['gender'] = df['gender'].fillna(df['pk_cid'].map(mode_gender_cid))
df['gender'] = df['gender'].fillna(df['pk_partition'].map(mode_gender_partition))
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])  # fallback global


In [214]:
df.select_dtypes(include=['number']).isnull().sum().sort_values(ascending=False)

region_code           2138
pk_cid                   0
em_account_pp            0
em_account_p             0
debit_card               0
emc_account              0
payroll_account          0
pension_plan             0
payroll                  0
credit_card              0
long_term_deposit        0
active_customer          0
securities               0
funds                    0
mortgage                 0
loans                    0
short_term_deposit       0
salary                   0
age                      0
em_acount                0
dtype: int64

In [215]:
df.select_dtypes(include=['object']).isnull().sum().sort_values(ascending=False)

pk_partition     0
entry_date       0
entry_channel    0
segment          0
country_id       0
gender           0
deceased         0
dtype: int64

#### dtypes transformation

In [216]:
df['pk_partition'] = pd.to_datetime(df['pk_partition'], errors='coerce')
df['entry_date'] = pd.to_datetime(df['entry_date'], errors='coerce')

## New interesting features

* z_months_since_entry

In [217]:
df['z_months_since_entry'] = ((df['pk_partition'] - df['entry_date']).dt.days / 30).round()

* z_pct_months_active

In [218]:
df = df.sort_values(['pk_cid', 'pk_partition'])

df['active_flag'] = df['active_customer'].eq(1).astype(int)
df['cum_months_total'] = df.groupby('pk_cid').cumcount() + 1
df['cum_months_active'] = df.groupby('pk_cid')['active_flag'].cumsum()

# Calculamos el porcentaje incremental
df['z_pct_months_active'] = (df['cum_months_active'] / df['cum_months_total']).round(2)
df = df.drop(columns=['active_flag', 'cum_months_total', 'cum_months_active'])

* z_segment_mode

In [219]:
df = df.sort_values(['pk_cid', 'pk_partition'])

# Detectar cambios de segmento
df['segment_prev'] = df.groupby('pk_cid')['segment'].shift(1)
df['z_partition_segment_changed'] = ((df['segment'] != df['segment_prev']) & df['segment_prev'].notna()).astype(int)


df['z_segment_mode'] = (
    df.groupby('pk_cid')['segment']
    .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

df = df.drop(columns=['segment_prev'])

* z_churn_customer_flag

In [220]:
df = df.sort_values(['pk_cid', 'pk_partition'])

df['next_partition'] = df.groupby('pk_cid')['pk_partition'].shift(-1)
last_partition = df['pk_partition'].max()

df['z_churn_customer_flag'] = np.where(
    df['next_partition'].isna() &
    (df['pk_partition'] != last_partition),
    1, 
    0
)

df = df.drop(columns=['next_partition'])

* z_new_customer_flag

In [221]:
df['z_new_customer_flag'] = (df['z_months_since_entry'] == 0).astype(int)

* z_num_products

In [222]:
df['z_num_products'] = df[product_cols].sum(axis=1)

In [223]:
df

,pk_cid,pk_partition,entry_date,entry_channel,active_customer,segment,country_id,region_code,gender,age,...,debit_card,em_account_p,em_acount,z_months_since_entry,z_pct_months_active,z_partition_segment_changed,z_segment_mode,z_churn_customer_flag,z_new_customer_flag,z_num_products
1479563,15891,2018-07-01,2018-07-01,KAT,1.0,03 - UNIVERSITARIO,ES,28.0,H,59,...,0,0,1,0.0,1.0,0,02 - PARTICULARES,0,1,1.0
2168122,15891,2018-08-01,2018-07-01,KAT,0.0,02 - PARTICULARES,ES,28.0,H,59,...,0,0,0,1.0,0.5,1,02 - PARTICULARES,1,0,0.0
2962973,16063,2018-11-01,2018-11-01,KAT,1.0,03 - UNIVERSITARIO,ES,28.0,H,62,...,0,0,0,0.0,1.0,0,02 - PARTICULARES,0,1,0.0
3628236,16063,2018-12-01,2018-11-01,KAT,1.0,02 - PARTICULARES,ES,28.0,H,62,...,0,0,0,1.0,1.0,1,02 - PARTICULARES,0,0,0.0
4028169,16063,2019-01-01,2018-11-01,KAT,1.0,02 - PARTICULARES,ES,28.0,H,62,...,0,0,0,2.0,1.0,0,02 - PARTICULARES,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5679916,1553685,2019-05-01,2019-05-01,KHE,0.0,03 - UNIVERSITARIO,ES,13.0,V,52,...,0,0,0,0.0,0.0,0,03 - UNIVERSITARIO,0,1,0.0
5679915,1553686,2019-05-01,2019-05-01,KHE,0.0,03 - UNIVERSITARIO,ES,41.0,H,30,...,0,0,0,0.0,0.0,0,03 - UNIVERSITARIO,0,1,0.0
5679914,1553687,2019-05-01,2019-05-01,KHE,0.0,03 - UNIVERSITARIO,ES,28.0,V,21,...,0,0,0,0.0,0.0,0,03 - UNIVERSITARIO,0,1,0.0
5679913,1553688,2019-05-01,2019-05-01,KHE,0.0,03 - UNIVERSITARIO,ES,39.0,H,43,...,0,0,0,0.0,0.0,0,03 - UNIVERSITARIO,0,1,0.0


In [224]:
df[df["z_num_products"] == 0]["active_customer"].value_counts()

active_customer
0.0    1059439
1.0      62068
Name: count, dtype: int64

In [225]:
df[(df["z_num_products"] == 0) & (df["active_customer"] == 1)]

,pk_cid,pk_partition,entry_date,entry_channel,active_customer,segment,country_id,region_code,gender,age,...,debit_card,em_account_p,em_acount,z_months_since_entry,z_pct_months_active,z_partition_segment_changed,z_segment_mode,z_churn_customer_flag,z_new_customer_flag,z_num_products
2962973,16063,2018-11-01,2018-11-01,KAT,1.0,03 - UNIVERSITARIO,ES,28.0,H,62,...,0,0,0,0.0,1.00,0,02 - PARTICULARES,0,1,0.0
3628236,16063,2018-12-01,2018-11-01,KAT,1.0,02 - PARTICULARES,ES,28.0,H,62,...,0,0,0,1.0,1.00,1,02 - PARTICULARES,0,0,0.0
4028169,16063,2019-01-01,2018-11-01,KAT,1.0,02 - PARTICULARES,ES,28.0,H,62,...,0,0,0,2.0,1.00,0,02 - PARTICULARES,0,0,0.0
4480637,16063,2019-02-01,2018-11-01,KAT,1.0,02 - PARTICULARES,ES,28.0,H,62,...,0,0,0,3.0,1.00,0,02 - PARTICULARES,0,0,0.0
5134317,16063,2019-04-01,2018-11-01,KAT,1.0,02 - PARTICULARES,ES,28.0,H,62,...,0,0,0,5.0,0.83,0,02 - PARTICULARES,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5679720,1552680,2019-05-01,2019-05-01,KHE,1.0,03 - UNIVERSITARIO,ES,28.0,H,51,...,0,0,0,0.0,1.00,0,03 - UNIVERSITARIO,0,1,0.0
5679765,1552696,2019-05-01,2019-05-01,KHE,1.0,03 - UNIVERSITARIO,ES,28.0,V,36,...,0,0,0,0.0,1.00,0,03 - UNIVERSITARIO,0,1,0.0
5679391,1552815,2019-05-01,2019-05-01,KHE,1.0,03 - UNIVERSITARIO,ES,28.0,V,33,...,0,0,0,0.0,1.00,0,03 - UNIVERSITARIO,0,1,0.0
5679415,1552858,2019-05-01,2019-05-01,KHE,1.0,03 - UNIVERSITARIO,ES,38.0,V,11,...,0,0,0,0.0,1.00,0,03 - UNIVERSITARIO,0,1,0.0


* z_active_with_product

In [226]:
# Creamos nueva variable de actividad 
# Si el cliente no tiene ningún producto realmente no es cliente activo
df['z_active_with_product'] = (~(df[product_cols] == 0).all(axis=1)).astype(int)

In [227]:
df[(df["z_num_products"] == 0) & (df["z_active_with_product"] == 1)]

,pk_cid,pk_partition,entry_date,entry_channel,active_customer,segment,country_id,region_code,gender,age,...,em_account_p,em_acount,z_months_since_entry,z_pct_months_active,z_partition_segment_changed,z_segment_mode,z_churn_customer_flag,z_new_customer_flag,z_num_products,z_active_with_product


### SALES

In [ ]:
#df_sales.rename(columns={"cid": "pk_cid", "month_sale": "pk_partition"}, inplace=True)
df_sales = df_sales.merge(df_product_description, how='left', left_on ="product_ID", right_on="pk_product_ID")
df_sales.drop(columns="pk_product_ID", inplace=True)

str_cols = ["pk_sale","cid", "product_ID"]
for col in str_cols:
    df_sales[col] = df_sales[col].astype(str)

df_sales['month_sale'] = pd.to_datetime(df_sales['month_sale'], errors='coerce')

In [248]:
df_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240773 entries, 0 to 240772
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   pk_sale         240773 non-null  object        
 1   cid             240773 non-null  object        
 2   month_sale      240773 non-null  datetime64[ns]
 3   product_ID      240773 non-null  object        
 4   net_margin      240773 non-null  float64       
 5   product_desc    240773 non-null  object        
 6   family_product  240773 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 12.9+ MB


In [249]:
df_sales

,pk_sale,cid,month_sale,product_ID,net_margin,product_desc,family_product
0,6666,33620,2018-05-01,2335,952.9,short_term_deposit,investment
1,6667,35063,2018-06-01,2335,1625.2,short_term_deposit,investment
2,6668,37299,2018-02-01,2335,1279.7,short_term_deposit,investment
3,6669,39997,2018-02-01,2335,1511.9,short_term_deposit,investment
4,6670,44012,2018-02-01,2335,1680.3,short_term_deposit,investment
...,...,...,...,...,...,...,...
240768,247434,1553456,2019-05-01,4657,56.7,em_acount,account
240769,247435,1553541,2019-05-01,4657,66.5,em_acount,account
240770,247436,1553559,2019-05-01,4657,73.0,em_acount,account
240771,247437,1553565,2019-05-01,4657,82.3,em_acount,account


In [250]:
# 16063, 1552680

In [251]:
df_sales[df_sales["cid"]=="16063"]

,pk_sale,cid,month_sale,product_ID,net_margin,product_desc,family_product


In [233]:
df_sales[df_sales["cid"]=="1552680"]

,pk_sale,cid,month_sale,product_ID,net_margin,product_desc,family_product


In [252]:
ventas_por_mes = (
    df_sales
    .groupby(['cid', 'month_sale'])['product_ID']
    .count()
    .reset_index(name='num_ventas')
)

ventas_por_mes[ventas_por_mes['num_ventas'] > 1]

,cid,month_sale,num_ventas
3,1000391,2018-02-01,2
15,1000468,2018-07-01,2
17,1000479,2018-10-01,3
23,1000502,2018-09-01,2
29,1000527,2019-01-01,2
...,...,...,...
202671,986296,2018-12-01,2
202674,991354,2019-02-01,4
202683,994005,2018-09-01,2
202685,994920,2018-04-01,2


In [253]:
ventas_por_mes.rename(columns={"cid":"pk_cid", "month_sale":"pk_partition","num_ventas":"z_num_purchased_products"}, inplace=True)

In [254]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5962924 entries, 1479563 to 5679912
Data columns (total 35 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   pk_cid                       object        
 1   pk_partition                 datetime64[ns]
 2   entry_date                   datetime64[ns]
 3   entry_channel                object        
 4   active_customer              float64       
 5   segment                      object        
 6   country_id                   object        
 7   region_code                  float64       
 8   gender                       object        
 9   age                          int64         
 10  deceased                     object        
 11  salary                       float64       
 12  short_term_deposit           int64         
 13  loans                        int64         
 14  mortgage                     int64         
 15  funds                        int64         
 16 

In [ ]:
df = df.merge(ventas_por_mes, how = "left", on = ["pk_cid", "pk_partition"])
df["z_num_purchased_products"] = df["z_num_purchased_products"].fillna(0)

In [258]:
df

,pk_cid,pk_partition,entry_date,entry_channel,active_customer,segment,country_id,region_code,gender,age,...,em_acount,z_months_since_entry,z_pct_months_active,z_partition_segment_changed,z_segment_mode,z_churn_customer_flag,z_new_customer_flag,z_num_products,z_active_with_product,z_num_purchased_products
0,15891,2018-07-01,2018-07-01,KAT,1.0,03 - UNIVERSITARIO,ES,28.0,H,59,...,1,0.0,1.0,0,02 - PARTICULARES,0,1,1.0,1,1.0
1,15891,2018-08-01,2018-07-01,KAT,0.0,02 - PARTICULARES,ES,28.0,H,59,...,0,1.0,0.5,1,02 - PARTICULARES,1,0,0.0,0,0.0
2,16063,2018-11-01,2018-11-01,KAT,1.0,03 - UNIVERSITARIO,ES,28.0,H,62,...,0,0.0,1.0,0,02 - PARTICULARES,0,1,0.0,0,0.0
3,16063,2018-12-01,2018-11-01,KAT,1.0,02 - PARTICULARES,ES,28.0,H,62,...,0,1.0,1.0,1,02 - PARTICULARES,0,0,0.0,0,0.0
4,16063,2019-01-01,2018-11-01,KAT,1.0,02 - PARTICULARES,ES,28.0,H,62,...,0,2.0,1.0,0,02 - PARTICULARES,0,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5962919,1553685,2019-05-01,2019-05-01,KHE,0.0,03 - UNIVERSITARIO,ES,13.0,V,52,...,0,0.0,0.0,0,03 - UNIVERSITARIO,0,1,0.0,0,0.0
5962920,1553686,2019-05-01,2019-05-01,KHE,0.0,03 - UNIVERSITARIO,ES,41.0,H,30,...,0,0.0,0.0,0,03 - UNIVERSITARIO,0,1,0.0,0,0.0
5962921,1553687,2019-05-01,2019-05-01,KHE,0.0,03 - UNIVERSITARIO,ES,28.0,V,21,...,0,0.0,0.0,0,03 - UNIVERSITARIO,0,1,0.0,0,0.0
5962922,1553688,2019-05-01,2019-05-01,KHE,0.0,03 - UNIVERSITARIO,ES,39.0,H,43,...,0,0.0,0.0,0,03 - UNIVERSITARIO,0,1,0.0,0,0.0
